# InverseOps — SwinIR Training on Colab

**Setup:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# 1. Verify GPU is available
!nvidia-smi

In [ ]:
# 2. Mount Google Drive (for persistent checkpoints + data)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clone repo and install
!git clone https://github.com/tyy0811/inverseops.git /content/inverseops
%cd /content/inverseops
!pip install -e . -q

In [ ]:
# 4. Set up data
# Option A: Copy FMD data from Google Drive
# Upload your data/raw/fmd/ folder to Drive first, then:
!mkdir -p data/raw
!cp -r /content/drive/MyDrive/inverseops_data/fmd data/raw/fmd

# Option B: Download directly (uncomment and adjust URL)
# !wget -q <YOUR_DATA_URL> -O fmd.zip && unzip -q fmd.zip -d data/raw/fmd

In [ ]:
# 5. Quick sanity check
!ls data/raw/fmd/ | head -10
!python -c "import torch; print(f'CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0)}')"

In [ ]:
# 6. Set output dir to Google Drive so checkpoints survive disconnects
OUTPUT_DIR = "/content/drive/MyDrive/inverseops_outputs/training"
!mkdir -p {OUTPUT_DIR}

In [ ]:
# 7. Train
!python scripts/run_training.py \
    --config configs/denoise_swinir.yaml \
    --output-dir {OUTPUT_DIR} \
    --no-wandb

In [ ]:
# 8. Resume training (if Colab disconnected)
# !python scripts/run_training.py \
#     --config configs/denoise_swinir.yaml \
#     --output-dir {OUTPUT_DIR} \
#     --resume {OUTPUT_DIR}/checkpoints/latest.pt \
#     --no-wandb

In [ ]:
# 9. Check results
import json
with open(f"{OUTPUT_DIR}/training_summary.json") as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))

In [ ]:
# 10. Run evaluation
!python scripts/run_evaluation.py \
    --checkpoint {OUTPUT_DIR}/checkpoints/best.pt \
    --data-dir data/raw/fmd \
    --output-dir {OUTPUT_DIR}/eval